# Lab 07: Cost & Token Analysis

**Goal:** Track token usage, calculate costs per model/user,
and design cost monitoring using LangFuse native features.

**What you'll learn:**
- How LangFuse automatically calculates cost from model and token counts
- Cost breakdown views: by model, user, conversation, prompt version, and time
- LangFuse v4 Cost Tracking API: observations with usage_details and cost_details
- Writing cost analysis queries to monitor spending

In [ ]:
import os
import shutil
import textwrap

WORKDIR = "/tmp/k8s-lab-12-07"

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)

## Step 1: Cost Tracking in LangFuse

In [ ]:
print("LangFuse automatically calculates cost from:")
print("  - Model name (from generation)")
print("  - Input tokens + Output tokens")
print("  - Built-in pricing table\n")

pricing = [
    ("gpt-4-turbo",             "$10.00",  "$30.00"),
    ("gpt-3.5-turbo",           "$0.50",   "$1.50"),
    ("groq/llama3-70b",         "$0.59",   "$0.79"),
    ("groq/llama3-8b",          "$0.05",   "$0.08"),
    ("text-embedding-3-small",  "$0.02",   "N/A"),
]

print(f"{'Model':<30} {'Input/1M tokens':<18} {'Output/1M tokens'}")
for model, inp, out in pricing:
    print(f"{model:<30} {inp:<18} {out}")

## Step 2: Cost Breakdown Views

In [ ]:
print("LangFuse provides cost analysis by:\n")

views = [
    ("By model",       "Which models cost the most?",      "gpt-4: $12.50, llama3-70b: $3.20"),
    ("By user",        "Which users consume most tokens?",  "alice: $5.10, bob: $2.30"),
    ("By trace/conv",  "Which conversations are expensive?", "session_42: $0.85 (10 turns)"),
    ("By prompt ver",  "Does new prompt cost less?",        "v1: $0.008/req, v2: $0.006/req"),
    ("Over time",      "Is cost trending up or down?",      "Daily: $15 -> $12 after optimization"),
]

print(f"{'View':<16} {'Question':<40} {'Example'}")
for view, question, example in views:
    print(f"{view:<16} {question:<40} {example}")

## Step 3: LangFuse Cost Tracking API

In [ ]:
print("LangFuse tracks cost natively through its Python SDK:\n")

api_code = textwrap.dedent("""\
    from langfuse import Langfuse
    from langfuse.langchain import CallbackHandler

    langfuse = Langfuse()

    # Option 1: Automatic cost tracking via CallbackHandler
    # LangFuse auto-captures model, tokens, and cost from LangChain
    handler = CallbackHandler()
    result = chain.invoke(query, config={
        \"callbacks\": [handler],
        \"metadata\": {\"langfuse_user_id\": \"alice\", \"langfuse_session_id\": \"sess_42\"},
    })

    # Option 2: Manual generation logging with explicit cost (v4 observations)
    with langfuse.start_as_current_observation(name=\"chat_request\") as span:
        with langfuse.start_as_current_observation(
            as_type=\"generation\",
            name=\"llm_call\",
            model=\"groq/llama3-70b\",
            input=[{\"role\": \"user\", \"content\": \"What is RAG?\"}],
        ) as generation:
            generation.update(
                output={\"role\": \"assistant\", \"content\": \"RAG is...\"},
                usage_details={\"input\": 150, \"output\": 250},
                # LangFuse auto-calculates cost, or set it explicitly:
                # cost_details={\"input\": 0.00009, \"output\": 0.0002, \"total\": 0.00029},
            )

    langfuse.flush()

    # Option 3: Query cost data via the API client
    traces = langfuse.api.trace.list(user_id=\"alice\")
    for t in traces.data:
        print(f\"  Trace: {t.name}  Cost: ${t.total_cost:.6f}\")
""")

for line in api_code.strip().split("\n"):
    print(f"    {line}")

## TODO 1 Solution: Cost Tracking Code

Write code that:
- Defines a `PRICING` dict (model -> input_per_1m, output_per_1m)
- Has a `calculate_cost(model, tokens_in, tokens_out)` function
- Opens a LangFuse observation with `start_as_current_observation`
- Logs a nested `as_type="generation"` observation with `usage_details` and `cost_details`
- Fetches traces with `langfuse.api.trace.list()` and prints `total_cost` per trace

In [ ]:
# SOLUTION: Cost tracking with PRICING dict and LangFuse generation logging
todo1_code = textwrap.dedent("""\
    from langfuse import Langfuse

    # Initialize LangFuse client
    langfuse = Langfuse()

    # Pricing dictionary: model -> (input_per_1M_tokens, output_per_1M_tokens)
    PRICING = {
        \"gpt-4-turbo\":            (10.00, 30.00),
        \"gpt-3.5-turbo\":          (0.50, 1.50),
        \"groq/llama3-70b\":        (0.59, 0.79),
        \"groq/llama3-8b\":         (0.05, 0.08),
        \"text-embedding-3-small\": (0.02, 0.00),
    }

    def calculate_cost(model, tokens_in, tokens_out):
        if model not in PRICING:
            return 0.0, 0.0
        input_price, output_price = PRICING[model]
        cost_in = (tokens_in / 1_000_000) * input_price
        cost_out = (tokens_out / 1_000_000) * output_price
        return cost_in, cost_out

    tokens_in, tokens_out = 500, 200
    model_name = \"groq/llama3-70b\"
    cost_in, cost_out = calculate_cost(model_name, tokens_in, tokens_out)

    # v4: observations replace trace()/generation(). The enclosing span is the trace.
    with langfuse.start_as_current_observation(name=\"cost_demo\") as span:
        trace_id = span.trace_id

        with langfuse.start_as_current_observation(
            as_type=\"generation\",
            name=\"llm_call\",
            model=model_name,
            input=[{\"role\": \"user\", \"content\": \"Explain RAG\"}],
        ) as generation:
            generation.update(
                output={\"role\": \"assistant\", \"content\": \"RAG stands for...\"},
                usage_details={\"input\": tokens_in, \"output\": tokens_out},
                cost_details={
                    \"input\": cost_in,
                    \"output\": cost_out,
                    \"total\": cost_in + cost_out,
                },
            )

    langfuse.flush()

    # Fetch traces and print total_cost per trace
    traces = langfuse.api.trace.list(user_id=\"alice\")
    for t in traces.data:
        print(f\"Trace: {t.name}  total_cost: ${t.total_cost:.6f}\")
""")

with open(os.path.join(WORKDIR, "cost_tracker.py"), "w") as f:
    f.write(todo1_code)

In [ ]:
checks1 = [
    ("Has PRICING dict",             "PRICING" in todo1_code),
    ("Has model pricing entry",      "gpt-4" in todo1_code.lower() or "llama" in todo1_code.lower()),
    ("Has calculate_cost function",  "def calculate" in todo1_code or "def calc" in todo1_code),
    ("Has 1_000_000 or 1000000",     "1_000_000" in todo1_code or "1000000" in todo1_code),
    ("Has Langfuse import",          "Langfuse" in todo1_code or "langfuse" in todo1_code),
    ("Has observation creation",     "start_as_current_observation" in todo1_code or "start_observation" in todo1_code),
    ("Has generation observation",   'as_type="generation"' in todo1_code),
    ("Has usage_details",            "usage_details" in todo1_code),
    ("Has cost_details or api list", "cost_details" in todo1_code or "api.trace.list" in todo1_code),
]

score1 = sum(1 for _, ok in checks1 if ok)
print(f"Validating ({score1}/{len(checks1)}):\n")
for name, ok in checks1:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")

## TODO 2 Solution: Cost Analysis Queries

Write LangFuse SDK code snippets for cost monitoring.

In [ ]:
queries = [
    {
        "purpose": "Total cost per model over the last 24 hours",
        "code": "___",
        "check_terms": ["api.trace.list", "model", "total_cost"],
    },
    {
        "purpose": "Token consumption breakdown (input vs output) by model",
        "code": "___",
        "check_terms": ["usage_details", "input", "output"],
    },
    {
        "purpose": "Average cost per user session",
        "code": "___",
        "check_terms": ["session", "cost", "len"],
    },
    {
        "purpose": "Identify traces exceeding a cost threshold ($0.01)",
        "code": "___",
        "check_terms": ["total_cost", "0.01", "trace"],
    },
]

# SOLUTION: Fill in LangFuse SDK code snippets
queries[0]["code"] = "traces = langfuse.api.trace.list(); cost_by_model = {}; [cost_by_model.update({o.model: cost_by_model.get(o.model, 0) + (t.total_cost or 0)}) for t in traces.data for o in (t.observations or [])]"
queries[1]["code"] = "for t in traces.data: [print(f'{o.model}: usage_details input={o.usage_details} output={o.output}') for o in (t.observations or [])]"
queries[2]["code"] = "session_costs = {}; [session_costs.update({t.session_id: session_costs.get(t.session_id, 0) + (t.total_cost or 0)}) for t in traces.data]; avg_cost = sum(session_costs.values()) / len(session_costs)"
queries[3]["code"] = "expensive = [trace for trace in traces.data if (trace.total_cost or 0) > 0.01]; print(f'Found {len(expensive)} traces over $0.01')"


In [ ]:
score2 = 0
for i, q in enumerate(queries, 1):
    if q["code"] == "___":
        status = "TODO"
    elif all(t.lower() in q["code"].lower() for t in q["check_terms"]):
        status = "PASS"
        score2 += 1
    else:
        status = "FAIL"
    print(f"  [{status}] {i}. {q['purpose']}")
    print(f"         Code: {q['code'][:80]}...")

print(f"\nScore: {score2}/{len(queries)}")

## Summary

Key concepts:
1. LangFuse auto-calculates cost from model + tokens
2. Cost views: by model, user, conversation, prompt version, time
3. LangFuse v4 API: observation logging with usage_details/cost_details, api.trace.list for analysis
4. Cost analysis: aggregate by model, user, session; alert on thresholds

In [ ]:
print(f"TODO 1: {score1}/{len(checks1)} cost tracking checks")
print(f"TODO 2: {score2}/{len(queries)} cost analysis queries")
print(f"\nFiles generated in {WORKDIR}/")

## Key Takeaways

- **LangFuse auto-calculates cost** from model name + token counts using its built-in pricing table
- **Cost breakdown views** let you analyze spending by model, user, conversation, prompt version, and over time
- **LangFuse v4 Cost Tracking API** supports automatic tracking via CallbackHandler, manual `start_as_current_observation(as_type="generation")` logging with `usage_details`/`cost_details`, and querying cost data via `langfuse.api.trace.list()`
- **Cost analysis queries** aggregate `total_cost` by model/user/session from `langfuse.api.trace.list().data` and set alerts for traces exceeding cost thresholds
- **PRICING dictionaries** with per-model input/output rates per 1M tokens are the foundation for programmatic cost calculation